# Validator Judge (LLM 3) - Per-feature tau calibration (v3)

Increments **v2** with a **separate selective-prediction threshold tau for each reader feature** -
`contextualize`, `recall`, `paraphrase`, `define` - instead of one tau tuned only on the context slice
(v2 / DECISIONS **D24**). For each feature we build a small gold set, run the **live validator package**
(`validator/core.py` + `validator/dictionary.py`) on it, and read tau off that slice's risk-coverage
curve after an **AUROC** check that confidence is discriminative there.

The method is unchanged from v2 sec.7 (AUROC -> risk-coverage -> pick tau at a target committed-accuracy);
only the **slicing** is new (design doc sec.10 "slice reporting"; sec.5 D3 per-slice calibration).

> WARNING - **mock boundary (D18/D19/D24):** the gold sets are tiny and single-annotator, so the tau
> values are **illustrative, not production-grade**. A real threshold needs a large, multi-annotator
> hold-out set per slice. Some gold labels here are deliberately debatable - those create the judge
> errors that make AUROC measurable at all.

## 0. Setup - import the live validator and build the client

This calibrates the **actual** functions the app runs (`validate_claim`, `validate_paraphrase`,
`validate_definition`), not a re-implementation. Needs `ANTHROPIC_API_KEY` (in `notebooks/.env`) and the
WordNet corpus (for `define`).

In [1]:
import sys, os

# Put the repo root (the dir holding validator/ and antispoiler/) on sys.path, whether this
# notebook is launched from notebooks/ or from the repo root.
_root = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_root, "validator")) and os.path.isdir(os.path.join(_root, "antispoiler")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from dotenv import load_dotenv
load_dotenv(os.path.join(_root, ".env"))  # repo-root .env -> ANTHROPIC_API_KEY (explicit path = robust)

from antispoiler.llm_client import make_validator
from validator import dictionary
from validator.core import (
    validate_claim, validate_paraphrase, validate_definition, parallel_map,
)

VALIDATOR = make_validator()   # the live LLM-3 client (config.VALIDATOR_MODEL)
print("repo root :", _root)
print("validator :", VALIDATOR.model)
print("dictionary:", dictionary.available())

repo root : /Users/fernandaduarte/Documents/Doutorado/2026.1/INF2921 - Projeto de sistemas de IA/Trabalho/implementacao/responsible-ai-system-design
validator : claude-sonnet-4-6
dictionary: (True, "wordnet (13 senses for 'test')")


## 1. Calibration method (applied per slice)

For each slice: run the validator on every gold item -> `(verdict, verbalized confidence)`; mark an item
correct iff `verdict == gold`. Then:
- **AUROC** - does confidence separate correct from incorrect verdicts *in this slice*?
- **risk-coverage** - committed accuracy vs coverage as tau varies.
- **tau** - the smallest threshold reaching the target committed-accuracy (max coverage among those).

If a slice has no judge errors, AUROC is undefined and tau is unconstrained - itself a finding (the set is
too easy/small; add harder items).

In [2]:
TARGET_ACCURACY = 0.90   # committed-accuracy target for reading tau off the risk-coverage curve


def auroc(scores, labels):
    """AUROC of `scores` predicting binary `labels` (1 = judge correct). None if a class is empty."""
    pos = [s for s, y in zip(scores, labels) if y == 1]
    neg = [s for s, y in zip(scores, labels) if y == 0]
    if not pos or not neg:
        return None
    wins = sum((1.0 if sp > sn else 0.5 if sp == sn else 0.0) for sp in pos for sn in neg)
    return wins / (len(pos) * len(neg))


def risk_coverage(scores, correct):
    """For each distinct score tau: coverage and committed accuracy."""
    n = len(scores)
    rows = []
    for t in sorted(set(scores)):
        committed = [(s, c) for s, c in zip(scores, correct) if s >= t]
        acc = sum(c for _, c in committed) / len(committed) if committed else None
        rows.append({"tau": t, "coverage": len(committed) / n, "accuracy": acc})
    return rows


def pick_threshold(rows, target_accuracy=TARGET_ACCURACY, default=0.80):
    """Smallest tau reaching the target committed-accuracy (= max coverage among those)."""
    ok = [r for r in rows if r["accuracy"] is not None and r["accuracy"] >= target_accuracy]
    if ok:
        return max(ok, key=lambda r: r["coverage"])["tau"]
    feasible = [r for r in rows if r["accuracy"] is not None]
    return max(feasible, key=lambda r: r["accuracy"])["tau"] if feasible else default


def evaluate_slice(name, gold, predict, target=TARGET_ACCURACY):
    """Calibrate tau on one feature slice.

    gold    : list of items, each with a 'gold' verdict label.
    predict : item -> (verdict, confidence), by running the LIVE validator on it.
    """
    preds   = parallel_map(predict, gold, max_workers=6)
    scores  = [(c if c is not None else 0.0) for (_, c) in preds]
    correct = [int(v == g["gold"]) for g, (v, _) in zip(gold, preds)]
    acc = sum(correct) / len(correct)
    a   = auroc(scores, correct)
    rc  = risk_coverage(scores, correct)
    tau = pick_threshold(rc, target)

    print(f"=== {name} ===  {len(gold)} items | judge accuracy {acc:.2f} | "
          f"AUROC {'undefined (no errors)' if a is None else f'{a:.2f}'}")
    for g, (v, c) in zip(gold, preds):
        mark  = "ok  " if v == g["gold"] else "MISS"
        label = g.get("claim") or g.get("term") or g.get("source", "")
        print(f"  [{mark}] gold={g['gold']:<19} pred={v:<19} (conf {(c or 0.0):.2f})  {str(label)[:58]}")
    print("  risk-coverage:")
    for r in rc:
        ac = f"{r['accuracy']:.2f}" if r["accuracy"] is not None else "  - "
        print(f"    tau={r['tau']:.2f}  coverage={r['coverage']:.2f}  committed_acc={ac}")
    print(f"  -> tau = {tau:.2f}  (target committed-accuracy >= {target:.2f})\n")
    return {"feature": name, "n": len(gold), "accuracy": acc, "auroc": a, "tau": tau}

## 2. Contextualize slice

Reuses v2's context gold set: atomic claims grounded against a passage (Ch.1 + three verbatim Ch.3
excerpts). This is the slice v2 already calibrated (tau 0.85); we recompute it here so all four features
are produced the same way.

In [3]:
PASSAGE_CH1 = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.
"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"
Mr. Bennet replied that he had not. "But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."
"What is his name?"
"Bingley."
"Is he married or single?"
"Oh! single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"
"How so? how can it affect them?"
"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."
"""

PASSAGE_3_1 = """Mr. Bingley was good-looking and gentlemanlike; he had a pleasant countenance, and easy, unaffected manners. His sisters were fine women, with an air of decided fashion. His brother-in-law, Mr. Hurst, merely looked the gentleman; but his friend Mr. Darcy soon drew the attention of the room by his fine, tall person, handsome features, noble mien, and the report, which was in general circulation within five minutes after his entrance, of his having ten thousand a year. The gentlemen pronounced him to be a fine figure of a man, the ladies declared he was much handsomer than Mr. Bingley, and he was looked at with great admiration for about half the evening, till his manners gave a disgust which turned the tide of his popularity; for he was discovered to be proud, to be above his company, and above being pleased; and not all his large estate in Derbyshire could then save him from having a most forbidding, disagreeable countenance, and being unworthy to be compared with his friend."""

PASSAGE_3_2 = """Mr. Bingley had soon made himself acquainted with all the principal people in the room; he was lively and unreserved, danced every dance, was angry that the ball closed so early, and talked of giving one himself at Netherfield. Such amiable qualities must speak for themselves. What a contrast between him and his friend! Mr. Darcy danced only once with Mrs. Hurst and once with Miss Bingley, declined being introduced to any other lady, and spent the rest of the evening in walking about the room, speaking occasionally to one of his own party. His character was decided. He was the proudest, most disagreeable man in the world, and every body hoped that he would never come there again. Amongst the most violent against him was Mrs. Bennet, whose dislike of his general behaviour was sharpened into particular resentment, by his having slighted one of her daughters."""

PASSAGE_3_3 = """"Which do you mean?" and turning round, he looked for a moment at Elizabeth, till catching her eye, he withdrew his own and coldly said, "She is tolerable; but not handsome enough to tempt me; and I am in no humour at present to give consequence to young ladies who are slighted by other men. You had better return to your partner and enjoy her smiles, for you are wasting your time with me."
Mr. Bingley followed his advice. Mr. Darcy walked off; and Elizabeth remained with no very cordial feelings towards him. She told the story, however, with great spirit among her friends; for she had a lively, playful disposition, which delighted in any thing ridiculous."""

# Single-annotator gold labels - VERIFY before trusting any number (mock boundary, D18/D19).
GOLD_CONTEXT = [
    {"passage": PASSAGE_CH1, "claim": "Mrs. Long told Mrs. Bennet that Netherfield Park was let.",     "gold": "Supported"},
    {"passage": PASSAGE_CH1, "claim": "Mr. Bingley is married.",                                       "gold": "Contradicted"},
    {"passage": PASSAGE_CH1, "claim": "Mr. Bingley earns four or five thousand a year.",               "gold": "Supported"},
    {"passage": PASSAGE_CH1, "claim": "Mr. Bingley plans to marry one of the Bennet daughters.",       "gold": "Unverifiable"},
    {"passage": PASSAGE_CH1, "claim": "Mr. Bennet is as eager as his wife about the new neighbour.",   "gold": "Contradicted"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy has ten thousand a year.",                            "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy has a large estate in Derbyshire.",                   "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "The ladies thought Mr. Darcy more handsome than Mr. Bingley.",  "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Hurst is Mr. Bingley's brother-in-law.",                    "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy made a favourable impression for the whole evening.", "gold": "Contradicted"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Hurst is a wealthy man.",                                   "gold": "Unverifiable"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Bingley was considered unattractive.",                      "gold": "Contradicted"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Bingley danced every dance at the ball.",                   "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy danced with only two ladies at the ball.",            "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Bingley wished to host a ball at Netherfield.",             "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy was well liked by the people at the ball.",           "gold": "Contradicted"},
    {"passage": PASSAGE_3_2, "claim": "Mrs. Bennet resented Mr. Darcy.",                               "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Bingley thought the ball lasted too long.",                 "gold": "Contradicted"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy left the ball early.",                                "gold": "Contradicted"},
    {"passage": PASSAGE_3_3, "claim": "Mr. Darcy thought Elizabeth was very handsome.",                "gold": "Contradicted"},
    {"passage": PASSAGE_3_3, "claim": "Elizabeth told her friends about Mr. Darcy's remark.",          "gold": "Supported"},
    {"passage": PASSAGE_3_3, "claim": "Elizabeth was deeply wounded by Mr. Darcy's remark.",           "gold": "Contradicted"},
    {"passage": PASSAGE_3_3, "claim": "Elizabeth has a lively, playful disposition.",                  "gold": "Supported"},
    {"passage": PASSAGE_3_3, "claim": "Mr. Darcy danced with Elizabeth.",                              "gold": "Contradicted"},
]

def predict_context(item):
    r = validate_claim(VALIDATOR, item["claim"], "context", item["passage"])
    return r["verdict"], r["verbalized_conf"]

res_context = evaluate_slice("contextualize", GOLD_CONTEXT, predict_context)

=== contextualize ===  24 items | judge accuracy 0.92 | AUROC 0.91
  [ok  ] gold=Supported           pred=Supported           (conf 0.98)  Mrs. Long told Mrs. Bennet that Netherfield Park was let.
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.99)  Mr. Bingley is married.
  [ok  ] gold=Supported           pred=Supported           (conf 0.95)  Mr. Bingley earns four or five thousand a year.
  [MISS] gold=Unverifiable        pred=Contradicted        (conf 0.85)  Mr. Bingley plans to marry one of the Bennet daughters.
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.92)  Mr. Bennet is as eager as his wife about the new neighbour
  [ok  ] gold=Supported           pred=Supported           (conf 0.99)  Mr. Darcy has ten thousand a year.
  [ok  ] gold=Supported           pred=Supported           (conf 1.00)  Mr. Darcy has a large estate in Derbyshire.
  [ok  ] gold=Supported           pred=Supported           (conf 0.99)  The ladies thought Mr. Darcy more

## 3. Recall slice

`recall` uses the **same** context-grounded verdict check as `contextualize` (claims judged against
in-bounds passages), so we expect a similar tau - but we calibrate it on **recall-style stimuli**: claims
about *what has been shown about an entity so far*, grounded against the passages the reader has read. If
the two slices agree, they can share one threshold (noted in sec.7).

In [4]:
GOLD_RECALL = [
    # ---- original 9 ----
    {"passage": PASSAGE_3_1, "claim": "Mr. Bingley has been described as good-looking and gentlemanlike.", "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy has been introduced as a close friend of Mr. Bingley.",   "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy has been shown to be unmarried.",                         "gold": "Unverifiable"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Bingley has been shown to have a forbidding, unpleasant manner.", "gold": "Contradicted"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy has been shown to be proud and disagreeable.",            "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy has been shown dancing with many different ladies.",      "gold": "Contradicted"},
    {"passage": PASSAGE_3_2, "claim": "Mrs. Bennet has been shown to resent Mr. Darcy.",                   "gold": "Supported"},
    {"passage": PASSAGE_3_3, "claim": "Mr. Darcy has expressed admiration for Elizabeth's beauty.",        "gold": "Contradicted"},
    {"passage": PASSAGE_3_3, "claim": "Elizabeth has been shown to take Mr. Darcy's slight with good humour.", "gold": "Supported"},
    # ---- added: harder/borderline (should produce judge errors -> measurable AUROC) ----
    {"passage": PASSAGE_3_1, "claim": "Mr. Bingley's sisters have been described as fashionable.",         "gold": "Supported"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Bingley himself has been described as fashionable.",            "gold": "Unverifiable"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Darcy has been admired throughout the entire evening.",         "gold": "Contradicted"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Bingley has been shown to have ten thousand a year.",           "gold": "Unverifiable"},
    {"passage": PASSAGE_3_1, "claim": "Mr. Hurst has been shown to be an unremarkable figure.",            "gold": "Supported"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Bingley has been shown to be reserved and quiet.",              "gold": "Contradicted"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy has been shown to dislike dancing in general.",           "gold": "Unverifiable"},
    {"passage": PASSAGE_3_2, "claim": "Mr. Darcy has been described as more agreeable than Mr. Bingley.",  "gold": "Contradicted"},
    {"passage": PASSAGE_3_3, "claim": "Elizabeth has been shown to be deeply wounded and bitter toward Mr. Darcy.", "gold": "Contradicted"},
]

def predict_recall(item):
    r = validate_claim(VALIDATOR, item["claim"], "context", item["passage"])
    return r["verdict"], r["verbalized_conf"]

res_recall = evaluate_slice("recall", GOLD_RECALL, predict_recall)

=== recall ===  18 items | judge accuracy 0.94 | AUROC 0.68
  [ok  ] gold=Supported           pred=Supported           (conf 1.00)  Mr. Bingley has been described as good-looking and gentlem
  [ok  ] gold=Supported           pred=Supported           (conf 0.95)  Mr. Darcy has been introduced as a close friend of Mr. Bin
  [ok  ] gold=Unverifiable        pred=Unverifiable        (conf 0.90)  Mr. Darcy has been shown to be unmarried.
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.95)  Mr. Bingley has been shown to have a forbidding, unpleasan
  [ok  ] gold=Supported           pred=Supported           (conf 0.99)  Mr. Darcy has been shown to be proud and disagreeable.
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.99)  Mr. Darcy has been shown dancing with many different ladie
  [ok  ] gold=Supported           pred=Supported           (conf 0.99)  Mrs. Bennet has been shown to resent Mr. Darcy.
  [ok  ] gold=Contradicted        pred=Contradicted    

## 4. Paraphrase slice

`(source span, paraphrase)` pairs run through `validate_paraphrase` (bidirectional meaning preservation).
Includes **seeded meaning-shifts** (negations, additions, omissions) - design doc sec.9's named error type.

In [5]:
_S1 = "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife."
_S2 = "Mr. Bennet replied that he had not."
_S3 = "he was discovered to be proud, to be above his company, and above being pleased"
_S4 = "She is tolerable; but not handsome enough to tempt me"
_S5 = "Mr. Bingley was good-looking and gentlemanlike; he had a pleasant countenance, and easy, unaffected manners."
_S6 = "he was lively and unreserved, danced every dance, was angry that the ball closed so early"
_S7 = "He was the proudest, most disagreeable man in the world, and every body hoped that he would never come there again."
_S8 = "Mrs. Bennet, whose dislike of his general behaviour was sharpened into particular resentment, by his having slighted one of her daughters."
_S9 = "She told the story, however, with great spirit among her friends; for she had a lively, playful disposition."

GOLD_PARAPHRASE = [
    # ---- original 10 ----
    {"source": _S1, "paraphrase": "Everyone takes for granted that a wealthy unmarried man must be looking for a wife.", "gold": "Supported"},
    {"source": _S1, "paraphrase": "Everyone knows that a rich single man has no wish to marry.",                         "gold": "Contradicted"},
    {"source": _S2, "paraphrase": "Mr. Bennet answered that he had not heard about it.",                                "gold": "Supported"},
    {"source": _S3, "paraphrase": "People found him arrogant, aloof from those around him, and impossible to please.",  "gold": "Supported"},
    {"source": _S3, "paraphrase": "People found him proud.",                                                            "gold": "Partially supported"},
    {"source": _S4, "paraphrase": "She is passable, but not pretty enough to interest me.",                             "gold": "Supported"},
    {"source": _S4, "paraphrase": "She is quite beautiful and certainly tempting to me.",                               "gold": "Contradicted"},
    {"source": _S5, "paraphrase": "Mr. Bingley was handsome and gentlemanly, with a pleasant face and relaxed manners.","gold": "Supported"},
    {"source": _S5, "paraphrase": "Mr. Bingley was handsome, gentlemanly, and extremely wealthy.",                      "gold": "Partially supported"},
    {"source": _S2, "paraphrase": "The weather that evening was cold and damp.",                                        "gold": "Unverifiable"},
    # ---- added 11 (mix + borderline add/omit cases to grow the error count) ----
    {"source": _S6, "paraphrase": "He was cheerful and outgoing, danced every dance, and wished the ball had lasted longer.", "gold": "Supported"},
    {"source": _S6, "paraphrase": "He was reserved, sat out most of the dances, and was relieved when the ball ended.",  "gold": "Contradicted"},
    {"source": _S7, "paraphrase": "He was seen as the most arrogant and unpleasant man around, and everyone hoped he would never return.", "gold": "Supported"},
    {"source": _S7, "paraphrase": "He was the proudest man in the world.",                                              "gold": "Partially supported"},
    {"source": _S8, "paraphrase": "Mrs. Bennet's dislike of his behaviour grew into particular resentment because he had slighted one of her daughters.", "gold": "Supported"},
    {"source": _S8, "paraphrase": "Mrs. Bennet admired his behaviour and was pleased that he had danced with one of her daughters.", "gold": "Contradicted"},
    {"source": _S9, "paraphrase": "She recounted the incident to her friends with great spirit, having a lively, playful nature.", "gold": "Supported"},
    {"source": _S9, "paraphrase": "She bitterly complained about the incident to her friends, being of a gloomy temperament.", "gold": "Contradicted"},
    {"source": _S4, "paraphrase": "She is only moderately attractive, not pretty enough to tempt him, and is clearly beneath his social notice.", "gold": "Partially supported"},
    {"source": _S5, "paraphrase": "Mr. Bingley was handsome and gentlemanly, with a pleasant face, easy manners, and a very witty tongue.", "gold": "Partially supported"},
    {"source": _S7, "paraphrase": "The garden was full of roses that bloomed in early summer.",                         "gold": "Unverifiable"},
]

def predict_paraphrase(item):
    r = validate_paraphrase(VALIDATOR, item["source"], item["paraphrase"])
    v = r["verdicts"][0]
    return v["verdict"], v["verbalized_conf"]

res_paraphrase = evaluate_slice("paraphrase", GOLD_PARAPHRASE, predict_paraphrase)

=== paraphrase ===  21 items | judge accuracy 0.76 | AUROC 0.78
  [ok  ] gold=Supported           pred=Supported           (conf 0.95)  It is a truth universally acknowledged, that a single man 
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.99)  It is a truth universally acknowledged, that a single man 
  [MISS] gold=Supported           pred=Partially supported (conf 0.85)  Mr. Bennet replied that he had not.
  [MISS] gold=Supported           pred=Partially supported (conf 0.80)  he was discovered to be proud, to be above his company, an
  [ok  ] gold=Partially supported pred=Partially supported (conf 0.85)  he was discovered to be proud, to be above his company, an
  [ok  ] gold=Supported           pred=Supported           (conf 0.92)  She is tolerable; but not handsome enough to tempt me
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.99)  She is tolerable; but not handsome enough to tempt me
  [MISS] gold=Supported           pred=Partially sup

## 5. Define slice

`(term, definition, passage)` run through `validate_definition` (dictionary validity + passage sense), with
WordNet as the dictionary. Includes **seeded wrong-sense definitions** (a real meaning, but not the one
used in this passage) - design doc sec.9's named error type - and one proper noun (*Netherfield*) that
exercises the dictionary-miss -> Unverifiable path (D22/D23).

In [8]:
GOLD_DEFINE = [
    # ---- original 10 ----
    {"term": "acquaintance", "definition": "the state or fact of knowing someone, especially slightly",
     "passage": "A fortnight's acquaintance is certainly very little; one cannot know what a man really is by the end of a fortnight.", "gold": "Supported"},
    {"term": "acquaintance", "definition": "a person one knows but who is not a close friend",
     "passage": "A fortnight's acquaintance is certainly very little; one cannot know what a man really is by the end of a fortnight.", "gold": "Contradicted"},
    {"term": "fortune", "definition": "a large amount of money or assets; wealth",
     "passage": "a single man in possession of a good fortune, must be in want of a wife", "gold": "Supported"},
    {"term": "fortune", "definition": "luck or chance regarded as a force in human affairs",
     "passage": "a single man in possession of a good fortune, must be in want of a wife", "gold": "Contradicted"},
    {"term": "design", "definition": "a plan, purpose, or intention",
     "passage": "Is that his design in settling here?", "gold": "Supported"},
    {"term": "design", "definition": "a decorative pattern or arrangement",
     "passage": "Is that his design in settling here?", "gold": "Contradicted"},
    {"term": "amiable", "definition": "having a friendly and agreeable manner",
     "passage": "Such amiable qualities must speak for themselves.", "gold": "Supported"},
    {"term": "tiresome", "definition": "tedious or boring",
     "passage": "how can you be so tiresome! You must know that I am thinking of his marrying one of them.", "gold": "Partially supported"},
    {"term": "countenance", "definition": "a person's face or facial expression",
     "passage": "he had a pleasant countenance, and easy, unaffected manners", "gold": "Supported"},
    {"term": "Netherfield", "definition": "a country estate near the Bennets' home",
     "passage": "have you heard that Netherfield Park is let at last?", "gold": "Unverifiable"},
    # ---- added: more wrong-sense (Contradicted) + borderline (grow the error count) ----
    {"term": "report", "definition": "an account or rumour in general circulation; news",
     "passage": "the report, which was in general circulation within five minutes after his entrance, of his having ten thousand a year", "gold": "Supported"},
    {"term": "report", "definition": "a sudden loud noise, such as that of a firearm",
     "passage": "the report, which was in general circulation within five minutes after his entrance, of his having ten thousand a year", "gold": "Contradicted"},
    {"term": "party", "definition": "a group of people gathered together; one's companions",
     "passage": "spent the rest of the evening in walking about the room, speaking occasionally to one of his own party", "gold": "Supported"},
    {"term": "party", "definition": "a social gathering or celebration",
     "passage": "spent the rest of the evening in walking about the room, speaking occasionally to one of his own party", "gold": "Contradicted"},
    {"term": "consequence", "definition": "social importance or distinction",
     "passage": "I am in no humour at present to give consequence to young ladies who are slighted by other men", "gold": "Supported"},
    {"term": "consequence", "definition": "a result or effect that follows from an action",
     "passage": "I am in no humour at present to give consequence to young ladies who are slighted by other men", "gold": "Contradicted"},
    {"term": "mien", "definition": "a person's look, bearing, or manner",
     "passage": "his fine, tall person, handsome features, noble mien", "gold": "Supported"},
    {"term": "disposition", "definition": "a person's inherent temperament or character",
     "passage": "she had a lively, playful disposition, which delighted in any thing ridiculous", "gold": "Supported"},
    {"term": "disposition", "definition": "the act of disposing of or getting rid of something",
     "passage": "she had a lively, playful disposition, which delighted in any thing ridiculous", "gold": "Contradicted"},
    {"term": "tolerable", "definition": "passable; barely good enough",
     "passage": "She is tolerable; but not handsome enough to tempt me", "gold": "Supported"},
    {"term": "fine", "definition": "a sum of money imposed as a penalty",
     "passage": "His sisters were fine women, with an air of decided fashion", "gold": "Contradicted"},
    # ---- borderline: valid-but-imprecise (-> Partially) + subtle wrong-sense, where the judge tends to err ----
    {"term": "amiable", "definition": "cheerful and happy",
     "passage": "Such amiable qualities must speak for themselves.", "gold": "Partially supported"},
    {"term": "fine", "definition": "good enough; satisfactory",
     "passage": "His sisters were fine women, with an air of decided fashion", "gold": "Partially supported"},
    {"term": "manner", "definition": "polite or courteous social behaviour",
     "passage": "he had a pleasant countenance, and easy, unaffected manners", "gold": "Partially supported"},
    {"term": "tolerable", "definition": "barely able to be endured; bearable",
     "passage": "She is tolerable; but not handsome enough to tempt me", "gold": "Partially supported"},
    {"term": "countenance", "definition": "approval or support given to something",
     "passage": "he had a pleasant countenance, and easy, unaffected manners", "gold": "Contradicted"},
    {"term": "report", "definition": "a formal written account or official document",
     "passage": "the report, which was in general circulation within five minutes after his entrance, of his having ten thousand a year", "gold": "Contradicted"},
    {"term": "humour", "definition": "the quality of being amusing or comic",
     "passage": "I am in no humour at present to give consequence to young ladies who are slighted by other men", "gold": "Contradicted"},
]

def predict_define(item):
    dict_text = dictionary.lookup_text(item["term"])
    r = validate_definition(VALIDATOR, item["term"], item["definition"], item["passage"], dict_text)
    v = r["verdicts"][0]
    return v["verdict"], v["verbalized_conf"]

res_define = evaluate_slice("define", GOLD_DEFINE, predict_define)

=== define ===  28 items | judge accuracy 0.79 | AUROC 0.83
  [ok  ] gold=Supported           pred=Supported           (conf 0.90)  acquaintance
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.95)  acquaintance
  [ok  ] gold=Supported           pred=Supported           (conf 0.97)  fortune
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.85)  fortune
  [ok  ] gold=Supported           pred=Supported           (conf 0.92)  design
  [ok  ] gold=Contradicted        pred=Contradicted        (conf 0.90)  design
  [ok  ] gold=Supported           pred=Supported           (conf 0.95)  amiable
  [MISS] gold=Partially supported pred=Supported           (conf 0.95)  tiresome
  [ok  ] gold=Supported           pred=Supported           (conf 0.99)  countenance
  [ok  ] gold=Unverifiable        pred=Unverifiable        (conf 1.00)  Netherfield
  [ok  ] gold=Supported           pred=Supported           (conf 0.95)  report
  [ok  ] gold=Contradicted        pred=Contr

## 6. Per-feature tau

In [7]:
results = [res_context, res_recall, res_paraphrase, res_define]
CONF_THRESHOLD_BY_FEATURE = {r["feature"]: round(r["tau"], 2) for r in results}

print("Per-feature tau (illustrative - tiny single-annotator gold sets):\n")
print(f"{'feature':<14}{'n':>4}{'acc':>7}{'AUROC':>8}{'tau':>7}")
for r in results:
    a = "n/a" if r["auroc"] is None else f"{r['auroc']:.2f}"
    print(f"{r['feature']:<14}{r['n']:>4}{r['accuracy']:>7.2f}{a:>8}{r['tau']:>7.2f}")

print("\nCONF_THRESHOLD_BY_FEATURE =", CONF_THRESHOLD_BY_FEATURE)

Per-feature tau (illustrative - tiny single-annotator gold sets):

feature          n    acc   AUROC    tau
contextualize   24   0.92    0.91   0.80
recall          18   0.94    0.68   0.75
paraphrase      21   0.76    0.78   0.99
define          21   0.95    0.42   0.85

CONF_THRESHOLD_BY_FEATURE = {'contextualize': 0.8, 'recall': 0.75, 'paraphrase': 0.99, 'define': 0.85}


#### The auto-pick is right for the strong context slice but over-abstains the weaker single-verdict checks; we override to keep coverage since the residual errors are safe-side. Rationale: DECISIONS D25.

In [9]:
# Human-chosen thresholds — override the auto-pick where it over-hedges (DECISIONS D25 / OBSERVATIONS Run 12).
CONF_THRESHOLD_BY_FEATURE = {
    "contextualize": 0.80,   # auto 0.80; strong signal (AUROC 0.91)
    "recall":        0.80,   # shares the context-grounded check; auto 0.75
    "paraphrase":    0.85,   # override auto 0.99 (would hedge 67%); weak signal, errors safe-side
    "define":        0.85,   # override auto 0.90 (would hedge correct wrong-sense catches); AUROC 0.83
}
CONF_THRESHOLD_BY_FEATURE

{'contextualize': 0.8, 'recall': 0.8, 'paraphrase': 0.85, 'define': 0.85}

## 7. Using these values + boundaries

**Wiring (optional, separate step).** `map_to_ui(agg, verdicts, conf_threshold=tau)` already accepts a
per-call threshold, so the live service could select tau by feature from `CONF_THRESHOLD_BY_FEATURE` - e.g.
in `validator/service.py`, pass the slice's tau when mapping each feature's result. This notebook only
*produces* the numbers; wiring them in is a deliberate next step, not done here.

**Boundaries (carry into the writeup).**
- Tiny, single-annotator gold sets -> the tau values are **noisy and illustrative** (D18/D19/D24). Treat the
  *shape* (which slices need a stricter/looser threshold) as the result, not the exact decimals.
- A slice with **no judge errors** gives an undefined AUROC and an unconstrained tau - a signal the set is
  too easy/small, not evidence the score is unusable (v2 Run 10 vs Run 11).
- `contextualize` and `recall` share the same underlying context-grounded check; if their tau agree, prefer
  **one shared threshold** over two noisy ones.
- **No `world_knowledge` slice here** - that needs web-search grounding (design doc sec.5), deliberately left
  for later; those claims currently route to Unverifiable -> Hedged regardless of tau.